# 04. SQL Joins: Inner, Left & Cross Joins: Beginner Guide

### 📝 SQL Execution Order for Multi-Table Joins:
```text
┌─ Execution Order ────────────────────────────────────────────────────────────┐
│ 1. FROM (Left Table) ➔ 2. ON (Match Key Predicates) ➔ 3. JOIN (Merge & Pad)  │
│ ➔ 4. WHERE (Filter Combined Dataset) ➔ 5. SELECT (Emit Chosen Attributes)    │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **04. SQL Joins: Inner, Left & Cross Joins**. Relational algebra enables combining independent relations horizontally through key constraints. This notebook covers matching intersecting tuples (`INNER JOIN`), preserving baseline datasets (`LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`), generating Cartesian permutations (`CROSS JOIN`), traversing hierarchical self-referential structures (Self-Joins), and Anti-Join optimization (`NOT EXISTS`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Intersecting Key Matching: `INNER JOIN ... ON`
- [x] 🔹 Baseline Preservation: `LEFT OUTER JOIN ... ON`
- [x] 🔹 Full Cartesian Permutations: `CROSS JOIN`
- [x] 🔹 Anti-Join Filtering: Isolating Unmatched Rows via `NOT EXISTS`
- [x] 🔍 Scenario: Multi-Table Customer Dispute Attribution & Merchant Exposure Analysis









In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Intersecting Records: `INNER JOIN`
- **What it does:** Evaluates join predicates across two relations and returns only tuples that satisfy the matching condition in both tables.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 INNER JOIN table_2 t2 ON t1.key = t2.key`
- **Dataset Application & Code Demonstration:** Joins `transactions` with `customers` on `customer_id`.


In [2]:
%%sql
SELECT 
    t.transaction_id,
    t.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier,
    t.transaction_amount
FROM transactions t
INNER JOIN customers c ON t.customer_id = c.customer_id
LIMIT 5;


,transaction_id,customer_id,customer_name,account_tier,transaction_amount
0,TX109326,C55082,Priya Rodriguez,Standard,607.78
1,TX106376,C76616,Paul Anderson,VIP,1819.11
2,TX103301,C65296,John Silva,Platinum,64.08
3,TX110701,C42098,Daniel Thompson,Standard,1025.73
4,TX103284,C97782,Margaret Robinson,Silver,772.74


### 🔹 Baseline Preservation: `LEFT OUTER JOIN`
- **What it does:** Preserves all tuples from the left table; matching right table attributes are populated, while unmatched records are padded with `NULL`.
- **Syntax:** `SELECT t1.cols, t2.cols FROM table_1 t1 LEFT JOIN table_2 t2 ON t1.key = t2.key`
- **Dataset Application & Code Demonstration:** Joins `customers` with `transactions` to inspect customer transaction activity.


In [3]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    t.transaction_id,
    t.transaction_amount
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
LIMIT 5;


,customer_id,customer_name,transaction_id,transaction_amount
0,C93810,Richard Sharma,TX103677,286.69
1,C93810,Richard Sharma,TX103724,442.73
2,C93810,Richard Sharma,TX104540,93.24
3,C93810,Richard Sharma,TX105403,1368.86
4,C93810,Richard Sharma,TX105559,1133.10


### 🔹 Cartesian Product: `CROSS JOIN`
- **What it does:** Computes the Cartesian product of two relations, pairing every row from the left table with every row from the right table ($M \times N$ rows).
- **Syntax:** `SELECT * FROM table_1 CROSS JOIN table_2`
- **Dataset Application & Code Demonstration:** Cross-joins distinct regions with distinct card types.


In [4]:
%%sql
SELECT 
    r.region,
    c.card_type
FROM (SELECT DISTINCT region FROM transactions WHERE region IS NOT NULL) r
CROSS JOIN (SELECT DISTINCT card_type FROM transactions WHERE card_type IS NOT NULL) c
LIMIT 8;


,region,card_type
0,North,Visa
1,North,Amex
2,North,Discover
3,North,MasterCard
4,West,Visa
5,West,Amex
6,West,Discover
7,West,MasterCard


### 🔹 Anti-Join Optimization: Isolating Unmatched Rows
- **What it does:** Identifies rows in a primary relation that have no corresponding matching records in a target relation.
- **Syntax:** `SELECT t1.* FROM t1 WHERE NOT EXISTS (SELECT 1 FROM t2 WHERE t2.key = t1.key)`
- **Dataset Application & Code Demonstration:** Finds customers who have zero recorded transactions in the system.


In [5]:
%%sql
SELECT 
    c.customer_id,
    (c.first_name || ' ' || c.last_name) AS customer_name,
    c.account_tier
FROM customers c
WHERE NOT EXISTS (
    SELECT 1 
    FROM transactions t 
    WHERE t.customer_id = c.customer_id
)
LIMIT 5;


,customer_id,customer_name,account_tier
0,C42738,Sandra Clark,VIP
1,C64934,Amara Gonzalez,VIP
2,C11068,Aarav Johnson,Silver
3,C62595,Emma Johnson,Platinum
4,C25907,Mark Kim,Standard


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Multi-Table Financial Settlement Attribution
- **Objective:** Combine transactions, customers, and merchants in a single 3-table join to calculate merchant dispute liability.
- **Approach:** Perform chained `INNER JOIN` operations and aggregate financial metrics.


In [6]:
%%sql
SELECT 
    m.merchant_id,
    m.merchant_name,
    m.category,
    COUNT(t.transaction_id) AS total_tx_count,
    ROUND(SUM(t.transaction_amount), 2) AS total_volume_usd
FROM merchants m
INNER JOIN transactions t ON m.merchant_id = t.merchant_id
GROUP BY m.merchant_id, m.merchant_name, m.category
ORDER BY total_volume_usd DESC
LIMIT 5;


,merchant_id,merchant_name,category,total_tx_count,total_volume_usd
0,M5128,Pulse Express,Food & Dining,40,50504.15
1,M6761,SilverLine Tech,Healthcare & Wellness,46,48765.54
2,M9618,Beacon Mart,Crypto & Digital Assets,45,46568.24
3,M5757,Global Hub,Grocery & Supermarket,40,45356.37
4,M6952,Nexus Retail,E-Commerce & Marketplaces,40,44978.49
